Simple Image Segmentation with MONAI and U-Net

Level: Beginner

This notebook teaches image segmentation — predicting a label for every pixel in an image, not just one label for the whole image. We use MONAI's U-Net architecture on synthetic images with simple shapes, so you can learn the core segmentation workflow quickly, without needing to download a large real-world dataset.

This builds directly on classification concepts (see the previous notebook) but introduces: pixel-level ground truth (masks), the U-Net architecture, Dice Loss, and the Dice score metric.

In [ ]:
!pip install -q "monai[pillow]" matplotlib

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.data.synthetic import create_test_image_2d
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose,
    EnsureChannelFirst,
    ScaleIntensity,
    RandRotate90,
    RandFlip,
)
from monai.utils import set_determinism

set_determinism(seed=0)
random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

1. Generate and Explore the Data

Instead of downloading images, we generate synthetic ones on the fly using MONAI's create_test_image_2d — each image contains a few random blob shapes with noise, paired with a clean ground-truth mask marking exactly which pixels belong to a shape.

In [ ]:
sample_image, sample_mask = create_test_image_2d(128, 128, num_objs=3, rad_max=20, noise_max=0.5, num_seg_classes=1)

print("Image shape:", sample_image.shape)
print("Mask shape:", sample_mask.shape)
print("Mask unique values:", np.unique(sample_mask))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_image, cmap="gray")
axes[0].set_title("Image (input)")
axes[0].axis("off")

axes[1].imshow(sample_mask, cmap="gray")
axes[1].set_title("Mask (ground truth)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

2. Build the Dataset

Unlike classification, segmentation needs a matching mask for every image. We apply separate transform pipelines to images and masks — we scale image intensity, but never scale the mask, since its pixel values (0 or 1) must stay exact.

In [ ]:
class SyntheticSegDataset(Dataset):
    """Generates synthetic image/mask pairs on the fly for segmentation."""
    def __init__(self, num_samples, image_transforms, mask_transforms):
        self.num_samples = num_samples
        self.image_transforms = image_transforms
        self.mask_transforms = mask_transforms

    def __len__(self):
        return self.num_samples

    def __getitem__(self, index):
        image, mask = create_test_image_2d(128, 128, num_objs=3, rad_max=20, noise_max=0.5, num_seg_classes=1)
        image = self.image_transforms(image)
        mask = self.mask_transforms(mask)
        return image, mask

In [ ]:
image_transforms = Compose([
    EnsureChannelFirst(channel_dim="no_channel"),
    ScaleIntensity(),
])

mask_transforms = Compose([
    EnsureChannelFirst(channel_dim="no_channel"),
])

train_ds = SyntheticSegDataset(num_samples=300, image_transforms=image_transforms, mask_transforms=mask_transforms)
val_ds = SyntheticSegDataset(num_samples=50, image_transforms=image_transforms, mask_transforms=mask_transforms)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_ds)} | Validation samples: {len(val_ds)}")

3. Build the Model

We use MONAI's UNet, the standard segmentation architecture — shaped like a "U," it shrinks the image down to understand context, then expands it back up to produce a full pixel-by-pixel prediction. We also use Dice Loss instead of Cross-Entropy Loss, since it directly measures how well predicted and true masks overlap.

In [ ]:
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceLoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
dice_metric = DiceMetric(include_background=True, reduction="mean")

print(model)

4. Train the Model

Same training pattern as before (predict → loss → backpropagate → update), but tracking Dice score instead of accuracy. We train for 20 epochs since segmentation typically needs more iterations to converge than classification.

In [ ]:
NUM_EPOCHS = 20

train_losses = []
val_dice_scores = []

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    dice_metric.reset()
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            dice_metric(y_pred=preds, y=masks)

    val_dice = dice_metric.aggregate().item()
    val_dice_scores.append(val_dice)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train loss: {avg_train_loss:.4f} | Val Dice: {val_dice:.4f}")

5. Evaluate and Visualize Predictions

A Dice score close to 1.0 means near-perfect overlap between predicted and true masks. Visualizing image, true mask, and predicted mask side by side is the clearest way to judge segmentation quality — numbers alone can hide where a model is getting things subtly wrong

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, NUM_EPOCHS + 1), train_losses, marker="o")
axes[0].set_title("Training Loss (Dice Loss)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(range(1, NUM_EPOCHS + 1), val_dice_scores, marker="o", color="green")
axes[1].set_title("Validation Dice Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice Score")

plt.tight_layout()
plt.show()

In [ ]:
model.eval()

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

with torch.no_grad():
    for i in range(4):
        image, mask = val_ds[i]
        image_input = image.unsqueeze(0).to(device)
        output = model(image_input)
        pred = (torch.sigmoid(output) > 0.5).float().cpu()

        axes[0, i].imshow(image[0], cmap="gray")
        axes[0, i].set_title("Image")
        axes[0, i].axis("off")

        axes[1, i].imshow(mask[0], cmap="gray")
        axes[1, i].set_title("True Mask")
        axes[1, i].axis("off")

        axes[2, i].imshow(pred[0, 0], cmap="gray")
        axes[2, i].set_title("Predicted Mask")
        axes[2, i].axis("off")

plt.tight_layout()
plt.show()